In [1]:
%pip install uv --quiet
!uv pip install pandas numpy plotly matplotlib scipy
!uv sync

Note: you may need to restart the kernel to use updated packages.


Using Python 3.12.3 environment at: C:\Users\logan\OneDrive\Documents\SeniorSpring\Adv Data Sci\Modern-Store-Of-Value\.venv
Checked 5 packages in 31ms
Resolved 147 packages in 2ms
Checked 143 packages in 17ms


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

In [3]:
stooq_tickers = {
    "Crypto ETFs": ["BITW", "IBIT", "ETHA"], 
    "Individual Stocks": ["NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"],
    "Sector ETFs": ["XLU"],
    "Broad Market ETFs": ["SPY", "VTI"],
    "Commodity ETFs (Metals)": ["GLD", "SLV", "PPLT", "PALL"],
    "Commodity ETFs (Agriculture)": ["WEAT", "SOYB", "DBA"]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [4]:
# Create flat mapping for Category
category_map = {ticker: cat for cat, ticks in stooq_tickers.items() for ticker in ticks}

with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
    data = processor.download(stooq_tickers, start=start_date, end=end_date)

for ticker, frame in data.items():
    data[ticker] = frame.assign(Ticker=ticker, Category=category_map.get(ticker, "Other"))

combined_data = pd.concat(data.values()).reset_index()
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

In [5]:
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
    category_map = processor.build_category_map(stooq_tickers)
    data = processor.download(
        stooq_tickers,
        start=start_date,
        end=end_date,
    )

for ticker, frame in data.items():
    data[ticker] = frame.assign(Ticker=ticker, Category=category_map[ticker])

combined_data = pd.concat(data.values()).reset_index()

# combined_data.head()
data["AAPL"].tail()

,Open,High,Low,Close,Volume,OpenInt,Ticker,Category
Date,,,,,,,,
2025-11-30,270.158,280.380,265.32,278.85,877813393,0,AAPL,Individual Stocks
2025-12-31,278.010,288.620,266.95,271.86,924529010,0,AAPL,Individual Stocks
2026-01-31,272.255,277.840,243.42,259.48,1040017271,0,AAPL,Individual Stocks
2026-02-28,260.030,280.905,255.45,264.18,988633102,0,AAPL,Individual Stocks
2026-03-26,262.410,266.530,246.00,252.89,763091456,0,AAPL,Individual Stocks


# Functions

In [6]:
def evaluate_crisis_performance(df_long):
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    
    # Widened windows to ensure 2+ monthly points are captured
    crisis_events = {
        "2022 Bear Market":    ("2021-12-31", "2022-10-31"),
        "2023 Banking Crisis": ("2023-02-28", "2023-05-31"),
        "2025 Tariff Shock":   ("2025-03-31", "2025-04-30"),
        "2026 Iran War":       ("2026-02-28", "2026-03-31")
    }

    results = []
    for name, (start, end) in crisis_events.items():
        window = pivot_df.loc[start:end]
        if len(window) < 2: continue
            
        for ticker in window.columns:
            series = window[ticker].dropna()
            if len(series) >= 2:
                ret = (series.iloc[-1] / series.iloc[0]) - 1
                results.append({'Crisis': name, 'Ticker': ticker, 'Return': ret * 100, 'Category': category_map[ticker]})

    res_df = pd.DataFrame(results)
    gold_rets = res_df[res_df['Ticker'] == 'GLD'].set_index('Crisis')['Return']
    res_df['Excess vs Gold (%)'] = res_df.apply(lambda x: x['Return'] - gold_rets.get(x['Crisis'], 0), axis=1)
    res_df['Success'] = np.where(res_df['Excess vs Gold (%)'] > 0, "YES", "NO")

    # Faceted bar chart matching Logan's style
    fig = px.bar(res_df, x="Ticker", y="Excess vs Gold (%)", color="Category", 
                 facet_col="Crisis", facet_col_wrap=2, template="plotly_white",
                 title="Success Metric: Excess Returns vs. Gold During Crisis Windows")
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()
    return res_df

In [7]:
def calculate_resilience_recovery(df_long):
    """
    Calculates the average recovery time for every asset after a 5% drop.
    Success = Faster recovery than Gold.
    """
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    
    recovery_stats = []
    for ticker in pivot_df.columns:
        prices = pivot_df[ticker].dropna()
        rolling_max = prices.cummax()
        drawdown = (prices - rolling_max) / rolling_max
        
        # Calculate how many weeks it stays below peak
        is_underwater = drawdown < 0
        # This is a simplified proxy for 'average recovery time'
        underwater_weeks = is_underwater.sum() 
        
        recovery_stats.append({'Ticker': ticker, 'Total Weeks Underwater': underwater_weeks})
        
    res_df = pd.DataFrame(recovery_stats).sort_values('Total Weeks Underwater')
    
    fig = px.bar(res_df, x='Ticker', y='Total Weeks Underwater', 
                 title="Resilience: Total Weeks Spent Below Previous Peak",
                 template="plotly_white")
    fig.show()

In [8]:
def discover_market_shocks(df_long, target_ticker='SPY', sigma_threshold=2):
    """
    Finds dates where the market drop was more than 2 Standard Deviations from normal.
    Use these dates to find 'Crisis Events' for your report.
    """
    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    rets = pivot_df[target_ticker].pct_change()
    
    mean = rets.mean()
    std = rets.std()
    
    # A 'Shock' is a drop worse than (Mean - 2*StdDev)
    shocks = rets[rets < (mean - sigma_threshold * std)]
    
    print(f"--- Systemic Shocks detected for {target_ticker} ---")
    for date, val in shocks.items():
        print(f"Shock Date: {date.date()} | Drop: {val*100:.2f}%")
        
    return shocks

In [9]:
evaluate_crisis_performance(combined_data)

c:\Users\logan\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




,Crisis,Ticker,Return,Category,Excess vs Gold (%),Success
0,2022 Bear Market,AAPL,-13.286146,Individual Stocks,-2.143189,NO
1,2022 Bear Market,AMD,-58.262682,Individual Stocks,-47.119725,NO
2,2022 Bear Market,AMZN,-38.554557,Individual Stocks,-27.411599,NO
3,2022 Bear Market,DBA,0.405063,Commodity ETFs (Agriculture),11.548021,YES
4,2022 Bear Market,GLD,-11.142957,Commodity ETFs (Metals),0.000000,NO
...,...,...,...,...,...,...
80,2026 Iran War,TSLA,-7.552607,Individual Stocks,9.627754,YES
81,2026 Iran War,VTI,-5.673466,Broad Market ETFs,11.506896,YES
82,2026 Iran War,WEAT,2.436863,Commodity ETFs (Agriculture),19.617225,YES
83,2026 Iran War,WMT,-4.509574,Individual Stocks,12.670788,YES


In [10]:
calculate_resilience_recovery(combined_data)

In [11]:
discover_market_shocks(combined_data)

--- Systemic Shocks detected for SPY ---
Shock Date: 2022-04-30 | Drop: -8.78%
Shock Date: 2022-06-30 | Drop: -8.25%
Shock Date: 2022-09-30 | Drop: -9.24%


Date
2022-04-30   -0.087770
2022-06-30   -0.082454
2022-09-30   -0.092441
Name: SPY, dtype: float64

# New Implementations

In [12]:
CRISIS_EVENTS = {
    "2022 Bear Market":    ("2021-12-31", "2022-10-31"),
    "2023 Banking Crisis": ("2023-02-28", "2023-05-31"),
    "2025 Tariff Shock":   ("2025-03-31", "2025-04-30"),
    "2026 Iran War":       ("2026-02-28", "2026-03-31")
}

RECOVERY_BUFFER_MONTHS = 12
LOOKBACK_DAYS = 90  # 90 calendar days (approx. 3 months) to find local pre-crisis peak

In [13]:
def calculate_drawdown_recovery_stress(df_long, crisis_events=CRISIS_EVENTS, recovery_buffer_months=RECOVERY_BUFFER_MONTHS, lookback_days=LOOKBACK_DAYS):
    """
    For each crisis window and each ticker, computes:
        - Local Pre-Crisis Peak: Max price in the 90 days prior to crisis start.
        - Max Drawdown: Worst drop from that local peak during the crisis window.
        - Time to Recover: Periods from trough back to local pre-crisis peak.
        - Stress Score: abs(Max Drawdown %) * Time to Recover
            - Lower = better store of value (shallow drop, fast recovery)
            - Higher = worse store of value (deep drop, slow/no recovery)
    """

    pivot_df = df_long.pivot(index='Date', columns='Ticker', values='Close')
    pivot_df.index = pd.to_datetime(pivot_df.index)

    # Optional mapping if you have category metadata stored elsewhere
    # Defaulting to 'Other' if no mapping dictionary is provided in scope
    # category_map = {} 

    results = []

    for crisis_name, (start, end) in crisis_events.items():
        start_dt = pd.to_datetime(start)
        end_dt   = pd.to_datetime(end)

        # 1. THE LOOKBACK FIX: Define the bounding window for the pre-crisis peak
        lookback_start = start_dt - pd.Timedelta(days=lookback_days)

        # Extend window for recovery
        buffer_end = end_dt + pd.DateOffset(months=recovery_buffer_months)
        buffer_end = min(buffer_end, pivot_df.index.max())

        full_window   = pivot_df.loc[start_dt:buffer_end]
        crisis_window = pivot_df.loc[start_dt:end_dt]

        if len(crisis_window) < 2:
            continue

        for ticker in pivot_df.columns:
            crisis_series = crisis_window[ticker].dropna()
            full_series   = full_window[ticker].dropna()

            if len(crisis_series) < 2:
                continue

            # ── Max Drawdown (Using Localized Peak) ──────────────────────
            # Only search for the peak between lookback_start and the crisis start_dt
            try:
                pre_crisis_peak = pivot_df[ticker].loc[lookback_start:start_dt].max()
            except ValueError:
                # Catch cases where the asset didn't exist in the lookback window
                continue
                
            if pd.isna(pre_crisis_peak):
                continue

            trough_val  = crisis_series.min()
            trough_date = crisis_series.idxmin()

            max_drawdown = (trough_val - pre_crisis_peak) / pre_crisis_peak

            if max_drawdown >= 0: # Asset went up or stayed flat
                results.append({
                    'Crisis': crisis_name,
                    'Ticker': ticker,
                    'Category': category_map.get(ticker, 'Other'),
                    'Max Drawdown (%)': 0.0,
                    'Time to Recover (periods)': 0,
                    'Never Recovered': False,
                    'Stress Score': 0.0,
                })
                continue

            # ── Time to Recover ──────────────────────────────────────────
            post_trough = full_series.loc[trough_date:]
            recovered   = post_trough[post_trough >= pre_crisis_peak]

            if len(recovered) == 0:
                never_recovered = True
                # Penalty: Time is capped at the maximum periods in the post-trough buffer
                time_to_recover = len(post_trough)
            else:
                never_recovered = False
                recovery_date   = recovered.index[0]
                time_to_recover = len(post_trough.loc[trough_date:recovery_date])

            # ── 2. THE METRIC FIX: Stress Score ──────────────────────────
            # Multiply severity by duration. Deep drops AND long times both compound the penalty.
            drawdown_pct = abs(max_drawdown * 100)
            stress_score = drawdown_pct * time_to_recover

            results.append({
                'Crisis': crisis_name,
                'Ticker': ticker,
                'Category': category_map.get(ticker, 'Other'),
                'Max Drawdown (%)': round(max_drawdown * 100, 2),
                'Time to Recover (periods)': time_to_recover,
                'Never Recovered': never_recovered,
                'Stress Score': round(stress_score, 2),
            })

    return pd.DataFrame(results)

In [14]:
def plot_drawdown_recovery(res_df):
    # ── Plot 1: Scatter ───────────────────────────────────────────────────
    fig1 = px.scatter(
        res_df,
        x='Max Drawdown (%)',
        y='Time to Recover (periods)',
        color='Category',
        text='Ticker',
        facet_col='Crisis',
        facet_col_wrap=2,
        template='plotly_white',
        title='Drawdown vs Recovery Time by Crisis'
    )
    fig1.update_traces(textposition='top center')
    fig1.show()

    # ── Plot 2: Ratio bar chart ───────────────────────────────────────────
    fig2 = px.bar(
        res_df,
        x='Ticker',
        y='Ratio',
        color='Category',
        facet_col='Crisis',
        facet_col_wrap=2,
        template='plotly_white',
        title='Recovery Ratio by Asset and Crisis (Lower = Better Store of Value)'
    )
    fig2.show()

In [15]:
# Check what prices MSFT and NVDA actually have in that window
pivot_df = combined_data.pivot(index='Date', columns='Ticker', values='Close')
pivot_df.index = pd.to_datetime(pivot_df.index)
print(pivot_df.loc["2023-02-28":"2023-05-31", ["MSFT", "NVDA", "GLD"]])

Ticker         MSFT     NVDA     GLD
Date                                
2023-02-28  246.009  23.1992  169.78
2023-03-31  284.355  27.7615  183.22
2023-04-30  303.055  27.7335  184.80
2023-05-31  324.604  37.8130  182.32


In [16]:
# 1. Calculate the drawdowns and stress scores
recovery_df = calculate_drawdown_recovery_stress(combined_data)

def summarize_and_normalize(res_df):
    """
    1. Aggregates the Stress Score to a single score per ticker.
    2. Normalizes to a 1-10 scale using Percentile Ranking where:
         1  = worst store of value (highest stress, lowest percentile)
         10 = best store of value  (lowest stress, highest percentile)
    """


    # ── Step 1: Single metric per ticker ─────────────────────────────────
    summary = (res_df
               .groupby(['Ticker', 'Category'])['Stress Score']
               .agg(
                   Mean_Stress='mean',
                   Median_Stress='median',
                   Crisis_Count='count'        
               )
               .reset_index())

    # Flag tickers that had at least one never-recovered crisis
    never_recovered_tickers = (res_df[res_df['Never Recovered'] == True]['Ticker'].unique())
    summary['Had_Never_Recovered'] = summary['Ticker'].isin(never_recovered_tickers)

    # ── Step 2: Percentile Normalize to 1–10 ─────────────────────────────
    # Lower stress = better. By setting ascending=False, the asset with the 
    # absolute lowest stress is ranked as the top percentile (1.0).
    pct_rank = summary['Mean_Stress'].rank(ascending=False, pct=True)
    
    min_rank = pct_rank.min()
    max_rank = pct_rank.max()

    # Prevent division by zero if all values are identical
    if max_rank == min_rank:
        summary['Score_1_10'] = 10.0
    else:
        summary['Score_1_10'] = (
            1 + 9 * (pct_rank - min_rank) / (max_rank - min_rank)  
        ).round(2)

    summary = summary.sort_values('Score_1_10', ascending=False).reset_index(drop=True)

    # ── Step 3: Visualize ─────────────────────────────────────────────────
    fig = px.bar(
        summary,
        x='Ticker',
        y='Score_1_10',
        color='Category',
        text='Score_1_10',
        template='plotly_white',
        title='Store of Value Score (1–10) — Based on Percentile Ranking (Crisis Stress)',
        labels={'Score_1_10': 'Score (10 = Top Percentile)'}
    )
    
    fig.update_traces(textposition='outside')
    
    # I adjusted the midline from 5.0 to 5.5, which is the true mathematical center of a 1 to 10 scale
    fig.add_hline(y=5.5, line_dash='dash', line_color='gray',
                  annotation_text='Midpoint (5.5)', annotation_position='top right')
    
    # Add a little headroom so the 10.0 bar isn't flush with the ceiling
    fig.update_layout(yaxis=dict(range=[0, 10.5]))
    
    fig.show()

    return summary
# Execute the function
summary_df = summarize_and_normalize(recovery_df)
summary_df

,Ticker,Category,Mean_Stress,Median_Stress,Crisis_Count,Had_Never_Recovered,Score_1_10
0,DBA,Commodity ETFs (Agriculture),7.7050,1.715,4,False,10.00
1,GLD,Commodity ETFs (Metals),18.1225,13.960,4,True,9.59
2,PPLT,Commodity ETFs (Metals),33.0750,28.260,4,True,9.18
3,XLU,Sector ETFs,39.1625,12.270,4,True,8.77
4,WMT,Individual Stocks,43.8925,32.460,4,True,8.36
5,BITW,Crypto ETFs,53.5100,53.510,1,True,7.95
6,SOYB,Commodity ETFs (Agriculture),56.1575,11.100,4,True,7.55
7,IBIT,Crypto ETFs,56.1850,56.185,2,True,7.14
8,JNJ,Individual Stocks,66.5750,20.110,4,True,6.73
9,SLV,Commodity ETFs (Metals),66.7350,26.965,4,True,6.32


### Export as csv

In [17]:
# ── Export Normalized Crisis Stress Metrics ───────────────────────────────
# Select the requested columns from the dataframe your function just returned
export_crisis_df = summary_df[['Ticker', 'Category', 'Mean_Stress', 'Score_1_10']].copy()

# Rename columns to make the CSV extremely clear for the team
export_crisis_df = export_crisis_df.rename(columns={
    'Ticker': 'Asset_Name',
    'Mean_Stress': 'Raw_Mean_Stress_Score',
    'Score_1_10': 'Normalized_Crisis_Score_1_10'
})

# Define the save path
crisis_output_path = "../data/crisis_metrics.csv"

# Export to CSV
export_crisis_df.to_csv(crisis_output_path, index=False)

print(f"Successfully exported {len(export_crisis_df)} assets to {crisis_output_path}")

# Display the top 5 rows so you can verify it worked
display(export_crisis_df.head())

Successfully exported 23 assets to ../data/crisis_metrics.csv


,Asset_Name,Category,Raw_Mean_Stress_Score,Normalized_Crisis_Score_1_10
0,DBA,Commodity ETFs (Agriculture),7.7050,10.00
1,GLD,Commodity ETFs (Metals),18.1225,9.59
2,PPLT,Commodity ETFs (Metals),33.0750,9.18
3,XLU,Sector ETFs,39.1625,8.77
4,WMT,Individual Stocks,43.8925,8.36
